# D114 — Create a Wheel

A wheel is a built Python distribution ending in `.whl`. It can contain our `movielens` package and metadata describing its external dependencies.

This lesson uses two external distributions:

- `simplejson`, imported by `JsonWriter`
- `python-dateutil`, imported as `dateutil` by `Rating`

We build three variants:

1. **All dependencies declared:** pip automatically installs both external distributions.
2. **Partial dependencies declared:** pip installs `simplejson`; the user must install `python-dateutil` separately.
3. **No dependencies declared:** the user must install both external distributions separately.

> `install_requires` does not copy external libraries inside our wheel. It writes `Requires-Dist` metadata. Pip downloads and installs those distributions separately. Physically copying another project into our package is called vendoring and is not demonstrated here.

D113 already created the complete `movielens-repo\movielens` application and its `simplejson` and `dateutil` imports. D114 does not modify application source files; it adds only packaging and build configuration.

## 1. Prepare the lesson directory

The build files belong beside the `movielens` package, not inside it. Run D113 first, then install the build frontend:

```bat
python -m pip install build
```

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

current_dir = Path.cwd()
notebook_dir = current_dir.parent if current_dir.name == "movielens-repo" else current_dir
lesson_root = notebook_dir / "movielens-repo"
os.chdir(lesson_root)
if not Path("movielens").is_dir():
    raise FileNotFoundError("Run D113 first to create movielens-repo\\movielens")
print("Build root:", Path.cwd())

## 2. Build-system configuration

`pyproject.toml` tells build tools to use setuptools. This file must be named exactly `pyproject.toml`.

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

## 3. Setup variant A: declare both dependencies

`find_packages()` discovers directories containing `__init__.py`. `install_requires` becomes dependency metadata. The console entry point creates a `movielens-report` command during installation.

In [ ]:
%%writefile setup_all_dependencies.py
from setuptools import find_packages, setup


setup(
    name="movielens-course",
    version="1.0.0",
    description="MovieLens modules teaching example",
    packages=find_packages(),
    install_requires=[
        "simplejson>=3.19,<4",
        "python-dateutil>=2.8,<3",
    ],
    entry_points={
        "console_scripts": [
            "movielens-report=movielens.__main__:main",
        ],
    },
)

Setuptools expects the active file to be named `setup.py`. Copy the teaching variant, clean old build metadata, and build only a wheel.

In [ ]:
def clean_build_artifacts():
    for path in (Path("build"), Path("movielens_course.egg-info")):
        if path.exists():
            shutil.rmtree(path)

shutil.copyfile("setup_all_dependencies.py", "setup.py")
clean_build_artifacts()

full_dist = Path(r"C:\tmp\movielens-wheel-lab\all-dependencies")
full_dist.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, "-m", "build", "--wheel",
    "--outdir", str(full_dist),
], check=True)

full_wheel = next(full_dist.glob("*.whl"))
print(full_wheel)

Command Prompt equivalent:

```bat
copy /Y setup_all_dependencies.py setup.py
python -m build --wheel --outdir C:\tmp\movielens-wheel-lab\all-dependencies
```

## 4. Inspect the wheel

A wheel is a ZIP archive. Package source is under `movielens/`; distribution metadata is under `movielens_course-1.0.0.dist-info/`.

In [ ]:
with zipfile.ZipFile(full_wheel) as wheel:
    names = wheel.namelist()
    metadata_name = next(name for name in names if name.endswith(".dist-info/METADATA"))
    metadata = wheel.read(metadata_name).decode("utf-8")

print("Files inside wheel:", len(names))
for name in names:
    print(name)

print("\nDependency metadata:")
for line in metadata.splitlines():
    if line.startswith("Requires-Dist:"):
        print(line)

The wheel contains neither a `simplejson/` nor a `dateutil/` directory. Their `Requires-Dist` lines tell pip to obtain separate wheels.

## 5. Setup variant B: omit one dependency

This variant declares `simplejson` but intentionally omits `python-dateutil`. The source still imports `dateutil`, so users must install it separately.

In [ ]:
%%writefile setup_partial_dependencies.py
from setuptools import find_packages, setup


setup(
    name="movielens-course",
    version="1.0.0",
    description="MovieLens modules teaching example",
    packages=find_packages(),
    install_requires=[
        "simplejson>=3.19,<4",
        # python-dateutil is intentionally not declared.
        # Install it separately: python -m pip install python-dateutil
    ],
    entry_points={
        "console_scripts": [
            "movielens-report=movielens.__main__:main",
        ],
    },
)

In [ ]:
shutil.copyfile("setup_partial_dependencies.py", "setup.py")
clean_build_artifacts()

partial_dist = Path(r"C:\tmp\movielens-wheel-lab\partial-dependencies")
partial_dist.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, "-m", "build", "--wheel",
    "--outdir", str(partial_dist),
], check=True)

partial_wheel = next(partial_dist.glob("*.whl"))
print(partial_wheel)

Command Prompt equivalent:

```bat
copy /Y setup_partial_dependencies.py setup.py
python -m build --wheel --outdir C:\tmp\movielens-wheel-lab\partial-dependencies
```

The two wheels have the same distribution name and version, so keep them in separate directories.

In [ ]:
with zipfile.ZipFile(partial_wheel) as wheel:
    metadata_name = next(
        name for name in wheel.namelist()
        if name.endswith(".dist-info/METADATA")
    )
    partial_metadata = wheel.read(metadata_name).decode("utf-8")

print("Declared dependencies:")
for line in partial_metadata.splitlines():
    if line.startswith("Requires-Dist:"):
        print(line)

## 6. Setup variant C: declare no dependencies

This variant has an empty `install_requires` list. Pip installs only `movielens-course`; both external distributions must be installed manually.

In [ ]:
%%writefile setup_no_dependencies.py
from setuptools import find_packages, setup


setup(
    name="movielens-course",
    version="1.0.0",
    description="MovieLens modules teaching example",
    packages=find_packages(),
    install_requires=[],
    entry_points={
        "console_scripts": [
            "movielens-report=movielens.__main__:main",
        ],
    },
)

In [ ]:
shutil.copyfile("setup_no_dependencies.py", "setup.py")
clean_build_artifacts()

no_deps_dist = Path(r"C:\tmp\movielens-wheel-lab\no-dependencies")
no_deps_dist.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, "-m", "build", "--wheel",
    "--outdir", str(no_deps_dist),
], check=True)

no_deps_wheel = next(no_deps_dist.glob("*.whl"))
print(no_deps_wheel)

Command Prompt equivalent:

```bat
copy /Y setup_no_dependencies.py setup.py
python -m build --wheel --outdir C:\tmp\movielens-wheel-lab\no-dependencies
```

In [ ]:
with zipfile.ZipFile(no_deps_wheel) as wheel:
    metadata_name = next(
        name for name in wheel.namelist()
        if name.endswith(".dist-info/METADATA")
    )
    no_deps_metadata = wheel.read(metadata_name).decode("utf-8")

no_deps_requirements = [
    line for line in no_deps_metadata.splitlines()
    if line.startswith("Requires-Dist:")
]
print("Declared dependencies:", no_deps_requirements)

## 7. Install and test the wheels

Normally, install the full-dependency wheel into a fresh virtual environment:

```bat
python -m venv C:\tmp\movielens-full-env
C:\tmp\movielens-full-env\Scripts\activate.bat
python -m pip install C:\tmp\movielens-wheel-lab\all-dependencies\movielens_course-1.0.0-py3-none-any.whl
python -m pip show movielens-course simplejson python-dateutil
movielens-report
```

Pip reads both `Requires-Dist` entries and installs three separate distributions.

For the partial-dependency wheel:

```bat
python -m venv C:\tmp\movielens-partial-env
C:\tmp\movielens-partial-env\Scripts\activate.bat
python -m pip install C:\tmp\movielens-wheel-lab\partial-dependencies\movielens_course-1.0.0-py3-none-any.whl
python -c "from movielens.models import Rating"
```

The import fails with `ModuleNotFoundError: No module named 'dateutil'` because it was not declared. Complete the manual installation:

```bat
python -m pip install python-dateutil
python -c "from movielens.models import Rating; print(Rating)"
movielens-report
```

## 8. Bundle the wheel and dependencies in a wheelhouse

A wheelhouse is a directory containing our wheel plus separate wheels for every dependency. This is the normal way to prepare an offline bundle:

```bat
mkdir C:\tmp\movielens-wheelhouse
python -m pip download --dest C:\tmp\movielens-wheelhouse C:\tmp\movielens-wheel-lab\all-dependencies\movielens_course-1.0.0-py3-none-any.whl
dir C:\tmp\movielens-wheelhouse\*.whl
```

The directory will contain `movielens-course`, `simplejson`, `python-dateutil`, and transitive dependency wheels such as `six`, depending on what the resolver requires.

Install the complete bundle without contacting an index:

```bat
python -m pip install --no-index --find-links C:\tmp\movielens-wheelhouse movielens-course
```

The dependencies travel together in one folder but remain separately maintained distributions.

## 9. Useful build and wheel commands

```bat
:: Build wheel and source distribution
python -m build

:: Build only a wheel
python -m build --wheel

:: List wheel contents
tar -tf C:\tmp\movielens-wheel-lab\all-dependencies\movielens_course-1.0.0-py3-none-any.whl

:: Install without contacting a package index; dependencies must already exist
python -m pip install --no-index --no-deps path\to\movielens_course-1.0.0-py3-none-any.whl

:: Uninstall the distribution
python -m pip uninstall movielens-course
```

`--no-deps` tells pip to ignore dependency metadata. Use it only when dependencies are already managed separately.

## 10. What is actually bundled?

| Item | Inside our wheel? | Installed automatically? |
|---|---:|---:|
| `movielens` modules | Yes | Yes |
| `simplejson` code | No | Yes, when declared |
| `dateutil` code | No | Yes only in the full variant |
| Dependency names and version ranges | In wheel metadata | Pip reads them |

### Summary

1. Put build configuration beside the import package.
2. Declare every required external distribution in normal production packages.
3. Run `python -m build --wheel` to create the wheel.
4. Inspect `.dist-info/METADATA` to see `Requires-Dist` entries.
5. Pip installs declared dependencies as separate packages; they are not embedded.
6. If a dependency is intentionally omitted, document and install it separately.
7. An empty `install_requires` list makes every external dependency a manual installation.